# Demo Chương 17: Sequence Labeling for Parts of Speech and Named Entities

### Nhóm 2:
- Võ Lê Ngọc Thịnh - 24521710
- Vũ Minh Phương - 24521421
- Nguyễn Hồng Phúc - 24521390
- Nguyễn Duy Khang - 24520755

## 1. Cài đặt Thư viện và Chuẩn bị Dữ liệu

Chúng ta sử dụng Brown Corpus với bộ nhãn Universal Tagset (17 nhãn) để đơn giản hóa quá trình phân tích nhưng vẫn đảm bảo tính bao quát của các từ loại.

In [5]:
!pip install sklearn_crfsuite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 63.9 MB/s eta 0:00:00


In [1]:
import nltk
import pandas as pd
from nltk.corpus import brown
from sklearn.metrics import accuracy_score, f1_score, classification_report

nltk.download('brown')
nltk.download('universal_tagset')

tagged_sentences = brown.tagged_sents(tagset='universal')

train_size = 40000
test_size = 10000
train_sents = tagged_sentences[:train_size]
test_sents = tagged_sentences[train_size:train_size + test_size]

print(f"Tổng số câu: {len(tagged_sentences)}")
print(f"Huấn luyện: {len(train_sents)} | Kiểm thử: {len(test_sents)}")

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.


Tổng số câu: 57340
Huấn luyện: 40000 | Kiểm thử: 10000


## 2. Mô hình Hidden Markov Model (HMM)

HMM là mô hình xác suất tính toán xác suất đồng thời $P(X, Y)$. Nó hoạt động dựa trên:
- Giả thuyết Markov: Xác suất của một nhãn chỉ phụ thuộc vào nhãn ngay trước đó.  
- Tính độc lập đầu ra: Một từ chỉ phụ thuộc vào nhãn trực tiếp tạo ra nó.

Chúng ta sử dụng thuật toán Viterbi (một kỹ thuật quy hoạch động) để tìm chuỗi nhãn có xác suất cao nhất cho một chuỗi từ.

In [2]:
from nltk.tag import hmm

print("Đang huấn luyện HMM (Generative Model)...")
hmm_model = hmm.HiddenMarkovModelTagger.train(train_sents)

def evaluate_hmm(model, test_data):
    y_true = [tag for sent in test_data for _, tag in sent]
    test_words = [[word for word, tag in sent] for sent in test_data]

    y_pred = []
    for words in test_words:
        y_pred.extend([tag for _, tag in model.tag(words)])
    return y_true, y_pred

hmm_y_true, hmm_y_pred = evaluate_hmm(hmm_model, test_sents)
print(f"HMM Accuracy: {accuracy_score(hmm_y_true, hmm_y_pred):.4f}")

Đang huấn luyện HMM (Generative Model)...
HMM Accuracy: 0.9332


## 3. Trích xuất Đặc trưng cho CRF (Feature Extraction)

Khác với HMM, CRF không bị giới hạn bởi các giả thuyết độc lập. Nó có thể sử dụng các Global Features được tổng hợp từ các Local Features tại mỗi vị trí $i$.

Chúng ta sẽ trích xuất các đặc trưng hình thái (Morphology) và ngữ cảnh (Context) như đã đề cập trong chương:
- Word shape: Viết hoa, chữ số, độ dài.  
- Affixes: Tiền tố (prefix) và hậu tố (suffix) để xử lý từ chưa biết (unknown words).
- Context: Thông tin của từ đứng trước và từ đứng sau.

In [3]:
def word2features(sent, i):
    word = sent[i][0]
    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word[-3:]': word[-3:],
        'word[-2:]': word[-2:],
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
    }

    if i > 0:
        word_prev = sent[i-1][0]
        features.update({
            '-1:word.lower()': word_prev.lower(),
            '-1:word.istitle()': word_prev.istitle(),
        })
    else:
        features['BOS'] = True


    if i < len(sent)-1:
        word_next = sent[i+1][0]
        features.update({
            '+1:word.lower()': word_next.lower(),
            '+1:word.istitle()': word_next.istitle(),
        })
    else:
        features['EOS'] = True

    return features

def sent2features(sent): return [word2features(sent, i) for i in range(len(sent))]
def sent2labels(sent): return [label for token, label in sent]

print("Đang trích xuất đặc trưng cho CRF...")
X_train = [sent2features(s) for s in train_sents]
y_train = [sent2labels(s) for s in train_sents]
X_test = [sent2features(s) for s in test_sents]
y_test = [sent2labels(s) for s in test_sents]

Đang trích xuất đặc trưng cho CRF...


## 4. Huấn luyện và Đánh giá CRF

CRF huấn luyện trực tiếp xác suất điều kiện $P(Y|X)$ thông qua các hàm log-linear. Trong NER, CRF thường được kết hợp với định dạng BIO tagging để nhận diện các thực thể nhiều từ.

In [6]:
import sklearn_crfsuite

crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1, c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)

print("Đang huấn luyện CRF (Discriminative Model)...")
crf.fit(X_train, y_train)

y_pred = crf.predict(X_test)
flat_y_test = [label for sent in y_test for label in sent]
flat_y_pred = [label for sent in y_pred for label in sent]

print(f"CRF Accuracy: {accuracy_score(flat_y_test, flat_y_pred):.4f}")

Đang huấn luyện CRF (Discriminative Model)...
CRF Accuracy: 0.9720


## 5. Tổng kết và So sánh Kết quả

Chúng ta đánh giá POS Tagging bằng Accuracy , trong khi với các bài toán như NER, chúng ta sẽ cần Precision, Recall và F1-score ở cấp độ thực thể.

In [7]:
results = pd.DataFrame({
    "Chỉ số": ["Accuracy", "Macro-F1"],
    "HMM": [accuracy_score(hmm_y_true, hmm_y_pred), f1_score(hmm_y_true, hmm_y_pred, average='macro')],
    "CRF": [accuracy_score(flat_y_test, flat_y_pred), f1_score(flat_y_test, flat_y_pred, average='macro')]
}).set_index("Chỉ số")

print("\n--- BẢNG SO SÁNH HIỆU NĂNG ---")
display(results)

print("\nChi tiết báo cáo phân loại cho CRF:")
print(classification_report(flat_y_test, flat_y_pred))


--- BẢNG SO SÁNH HIỆU NĂNG ---


,HMM,CRF
Chỉ số,,
Accuracy,0.933179,0.971991
Macro-F1,0.850724,0.925089



Chi tiết báo cáo phân loại cho CRF:
              precision    recall  f1-score   support

           .       1.00      1.00      1.00     23492
         ADJ       0.90      0.89      0.90      7644
         ADP       0.96      0.98      0.97     15516
         ADV       0.93      0.92      0.92      8654
        CONJ       0.99      0.99      0.99      4632
         DET       0.99      0.99      0.99     17044
        NOUN       0.96      0.97      0.97     28975
         NUM       0.97      0.98      0.97      1072
        PRON       0.99      0.98      0.99     11216
         PRT       0.95      0.91      0.93      5839
        VERB       0.98      0.98      0.98     27429
           X       0.52      0.47      0.49       116

    accuracy                           0.97    151629
   macro avg       0.93      0.92      0.93    151629
weighted avg       0.97      0.97      0.97    151629



Nhận xét:
- CRF thường cho độ chính xác cao hơn (~97%) vì nó có khả năng "nhìn" toàn bộ câu và sử dụng các đặc trưng tùy biến.
- HMM gặp khó khăn với các từ chưa biết (unknown words) do nó chỉ dựa vào xác suất phát xạ $P(word|tag)$.
- Việc sử dụng ngữ cảnh lân cận trong CRF giúp giải quyết các từ đa nghĩa (ambiguous words) như "back" (có thể là JJ, NN, VB, v.v.) hiệu quả hơn